# Local data inspection ? 2026-09-21
Read-only checks of local tracker.db. No upstream fetch performed. Dates reflect stored source values.

In [1]:
import sqlite3
from pathlib import Path
root = Path.cwd()
if not (root / "data/tracker.db").exists():
    root = root.parent
connection = sqlite3.connect((root / "data/tracker.db").resolve().as_uri() + "?mode=ro", uri=True)
queries = {
    "counts": "SELECT 'prices', count(*) FROM price_bars UNION ALL SELECT 'announcements', count(*) FROM announcements UNION ALL SELECT 'scores',count(*) FROM score_snapshots UNION ALL SELECT 'signals',count(*) FROM signals",
    "stock_coverage": "SELECT s.code,s.active,count(p.id),min(p.date),max(p.date) FROM stocks s LEFT JOIN price_bars p ON p.stock_id=s.id GROUP BY s.id",
    "announcements": "SELECT count(*),count(distinct stock_id),min(ann_date),max(ann_date),sum(ai_summary IS NOT NULL AND ai_summary != '') FROM announcements",
    "instruments": "SELECT instrument,count(*),min(date),max(date) FROM commodity_bars GROUP BY instrument",
    "signals": "SELECT source,count(*),min(date),max(date) FROM signals GROUP BY source",
    "runs": "SELECT id,run_at,finished_at,trigger,status FROM pipeline_runs ORDER BY id DESC LIMIT 5",
    "duplicate_prices": "SELECT count(*) FROM (SELECT stock_id,date FROM price_bars GROUP BY stock_id,date HAVING count(*)>1)",
    "duplicate_announcements": "SELECT count(*) FROM (SELECT stock_id,ann_id FROM announcements GROUP BY stock_id,ann_id HAVING count(*)>1)",
    "invalid_prices": "SELECT count(*) FROM price_bars WHERE close<=0 OR close IS NULL OR volume<0",
    "foreign_keys": "PRAGMA foreign_key_check"
}
for name, query in queries.items():
    print(name, connection.execute(query).fetchall())
connection.close()


counts [('prices', 16834), ('announcements', 518), ('scores', 175), ('signals', 669)]
stock_coverage [('PDI', 1, 563, '2024-07-03', '2026-09-18'), ('MEK', 1, 563, '2024-07-03', '2026-09-18'), ('SMI', 1, 563, '2024-07-03', '2026-09-18'), ('BC8', 1, 563, '2024-07-03', '2026-09-18'), ('AAR', 1, 563, '2024-07-03', '2026-09-18'), ('MAU', 0, 0, None, None), ('AUC', 1, 563, '2024-07-03', '2026-09-18'), ('CYL', 1, 563, '2024-07-03', '2026-09-18'), ('TTM', 1, 563, '2024-07-03', '2026-09-18'), ('HCH', 1, 563, '2024-07-03', '2026-09-18'), ('XAM', 0, 0, None, None), ('CNB', 1, 563, '2024-07-03', '2026-09-18'), ('CVV', 1, 563, '2024-07-03', '2026-09-18'), ('A1M', 1, 563, '2024-07-03', '2026-09-18'), ('WC8', 1, 563, '2024-07-03', '2026-09-18'), ('GL1', 1, 563, '2024-07-03', '2026-09-18'), ('GT1', 1, 563, '2024-07-03', '2026-09-18'), ('AGY', 1, 563, '2024-07-03', '2026-09-18'), ('VUL', 1, 563, '2024-07-03', '2026-09-18'), ('DYL', 1, 563, '2024-07-03', '2026-09-18'), ('LOT', 1, 563, '2024-07-03', '202